# Exploratory Data Analysis (EDA) — Pima Indians Diabetes Dataset

**Tujuan Notebook ini:**
- Memahami struktur dan karakteristik dataset
- Mendeteksi dan menangani missing value yang tersembunyi (nilai 0 tidak valid)
- Memvisualisasikan distribusi setiap fitur
- Menganalisis korelasi antar fitur
- Mempersiapkan data untuk pemodelan ML

---

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Style global untuk semua plot
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['font.size'] = 11
PALETTE = ['#378ADD', '#E24B4A']  # Biru = tidak diabetes, Merah = diabetes

print('✅ Library berhasil diimport!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   seaborn : {sns.__version__}')

## 2. Load Dataset

In [ ]:
# Sesuaikan path sesuai lokasi file kamu
df = pd.read_csv('../data/diabetes.csv')

print(f'Jumlah baris   : {df.shape[0]}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'Kolom          : {list(df.columns)}')
print()
df.head(10)

## 3. Informasi Umum Dataset

In [ ]:
print('=== INFO DATASET ===')
df.info()
print()
print('=== TIPE DATA ===')
print(df.dtypes)

In [ ]:
print('=== STATISTIK DESKRIPTIF ===')
df.describe().round(2)

In [ ]:
# Distribusi target (Outcome)
outcome_counts = df['Outcome'].value_counts()
outcome_pct    = df['Outcome'].value_counts(normalize=True) * 100

print('=== DISTRIBUSI TARGET (Outcome) ===')
print(f'Tidak Diabetes (0) : {outcome_counts[0]:>4} pasien  ({outcome_pct[0]:.1f}%)')
print(f'Diabetes       (1) : {outcome_counts[1]:>4} pasien  ({outcome_pct[1]:.1f}%)')
print()
print('>>> Dataset sedikit imbalanced — kelas 0 lebih banyak dari kelas 1.')
print('    Perhatikan metrik F1-Score dan Recall saat evaluasi model, bukan hanya Accuracy.')

## 4. Deteksi & Penanganan Missing Value

> **Catatan Penting:** Dataset ini tidak memiliki nilai `NaN`, namun beberapa kolom mengandung nilai `0` yang secara medis tidak mungkin terjadi. Nilai `0` ini adalah *missing value yang tersembunyi* dan perlu ditangani sebelum pemodelan.

In [ ]:
# --- Cek NaN biasa ---
print('=== CEK NaN (MISSING VALUE STANDAR) ===')
nan_counts = df.isnull().sum()
print(nan_counts)
print(f'\nTotal NaN: {nan_counts.sum()} — Tidak ada missing value standar.')

In [ ]:
# --- Deteksi nilai 0 yang tidak valid secara medis ---
# Kolom ini tidak boleh bernilai 0 pada manusia hidup
zero_not_valid = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('=== NILAI 0 YANG TIDAK VALID (HIDDEN MISSING VALUES) ===')
print(f'{"Kolom":<20} {"Jumlah 0":>10} {"Persentase":>12}')
print('-' * 45)
for col in zero_not_valid:
    count = (df[col] == 0).sum()
    pct   = count / len(df) * 100
    flag  = ' ⚠️  Tinggi!' if pct > 20 else ''
    print(f'{col:<20} {count:>10} {pct:>11.1f}%{flag}')

In [ ]:
# --- Visualisasi sebelum penanganan ---
fig, axes = plt.subplots(1, len(zero_not_valid), figsize=(18, 4))
fig.suptitle('Distribusi Kolom SEBELUM Penanganan Missing Value\n(nilai 0 yang tidak valid ditandai merah)',
             fontsize=13, fontweight='bold', y=1.02)

for i, col in enumerate(zero_not_valid):
    ax = axes[i]
    # Histogram normal
    non_zero = df[col][df[col] != 0]
    zero_val = df[col][df[col] == 0]
    ax.hist(non_zero, bins=25, color='#378ADD', alpha=0.8, label='Valid')
    ax.axvline(0, color='#E24B4A', linewidth=2.5, linestyle='--', label=f'0 = {len(zero_val)} baris')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Nilai')
    ax.set_ylabel('Frekuensi' if i == 0 else '')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot_before_imputation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Penanganan: Ganti nilai 0 dengan MEDIAN per kelompok Outcome ---
# Menggunakan median per kelas lebih baik daripada median global
# karena distribusi antara pasien diabetes dan tidak diabetes berbeda

df_clean = df.copy()

for col in zero_not_valid:
    # Hitung median per kelas
    median_0 = df_clean.loc[(df_clean['Outcome'] == 0) & (df_clean[col] != 0), col].median()
    median_1 = df_clean.loc[(df_clean['Outcome'] == 1) & (df_clean[col] != 0), col].median()
    
    # Ganti nilai 0 berdasarkan kelas
    mask_0 = (df_clean[col] == 0) & (df_clean['Outcome'] == 0)
    mask_1 = (df_clean[col] == 0) & (df_clean['Outcome'] == 1)
    df_clean.loc[mask_0, col] = median_0
    df_clean.loc[mask_1, col] = median_1
    
    print(f'{col:<20} | Median kelas 0: {median_0:.1f}  | Median kelas 1: {median_1:.1f}')

print()
print('✅ Penanganan selesai! Cek apakah masih ada 0:')
for col in zero_not_valid:
    remaining = (df_clean[col] == 0).sum()
    status = '✅ OK' if remaining == 0 else f'⚠️  masih ada {remaining}'
    print(f'   {col:<20}: {status}')

## 5. Visualisasi Distribusi Setiap Fitur

In [ ]:
# --- Distribusi semua fitur (histogram + KDE) berdasarkan kelas ---
features = [col for col in df_clean.columns if col != 'Outcome']

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle('Distribusi Setiap Fitur Berdasarkan Status Diabetes\n(setelah penanganan missing value)',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

labels = {0: 'Tidak Diabetes', 1: 'Diabetes'}

for i, col in enumerate(features):
    ax = axes[i]
    for outcome, color in zip([0, 1], PALETTE):
        subset = df_clean[df_clean['Outcome'] == outcome][col]
        ax.hist(subset, bins=25, alpha=0.55, color=color,
                label=labels[outcome], density=True)
        # Tambahkan KDE
        kde_x = np.linspace(subset.min(), subset.max(), 200)
        kde   = stats.gaussian_kde(subset)
        ax.plot(kde_x, kde(kde_x), color=color, linewidth=2)
    
    # Median lines
    med0 = df_clean[df_clean['Outcome'] == 0][col].median()
    med1 = df_clean[df_clean['Outcome'] == 1][col].median()
    ax.axvline(med0, color=PALETTE[0], linestyle='--', linewidth=1.2, alpha=0.8)
    ax.axvline(med1, color=PALETTE[1], linestyle='--', linewidth=1.2, alpha=0.8)
    
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.set_xlabel('Nilai')
    ax.set_ylabel('Densitas')
    ax.legend(fontsize=9)

# Hapus subplot kosong (subplot ke-9 tidak dipakai)
axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig('plot_distribusi_fitur.png', dpi=120, bbox_inches='tight')
plt.show()

print('Insight: Garis putus-putus menunjukkan median masing-masing kelas.')
print('Semakin jauh jarak antar garis, semakin penting fitur tersebut untuk prediksi.')

In [ ]:
# --- Boxplot untuk deteksi outlier ---
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
fig.suptitle('Boxplot Setiap Fitur (Deteksi Outlier)\nBiru = Tidak Diabetes | Merah = Diabetes',
             fontsize=13, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    data_to_plot = [
        df_clean[df_clean['Outcome'] == 0][col].values,
        df_clean[df_clean['Outcome'] == 1][col].values
    ]
    bp = ax.boxplot(data_to_plot, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(col, fontweight='bold')
    ax.set_xticklabels(['Tidak\nDiabetes', 'Diabetes'], fontsize=9)

plt.tight_layout()
plt.savefig('plot_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Analisis Korelasi

In [ ]:
# --- Heatmap Korelasi ---
corr_matrix = df_clean.corr()

fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle('Heatmap Korelasi Antar Fitur\n(nilai mendekati 1 atau -1 = korelasi kuat)',
             fontsize=13, fontweight='bold')

# Mask untuk segitiga atas (menghindari duplikasi)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor='white',
    square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Koefisien Korelasi'},
    ax=ax
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('plot_heatmap_korelasi.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Ranking korelasi fitur terhadap Outcome ---
corr_with_outcome = df_clean.corr()['Outcome'].drop('Outcome').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors  = ['#378ADD' if v >= 0 else '#E24B4A'
           for v in df_clean.corr()['Outcome'].drop('Outcome').loc[corr_with_outcome.index]]

bars = ax.barh(corr_with_outcome.index, corr_with_outcome.values,
               color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, corr_with_outcome.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Korelasi Absolut dengan Outcome', fontsize=11)
ax.set_title('Ranking Korelasi Fitur terhadap Diabetes (Outcome)\nSemakin besar = semakin berpengaruh',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, corr_with_outcome.max() + 0.07)
ax.invert_yaxis()
ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig('plot_korelasi_ranking.png', dpi=120, bbox_inches='tight')
plt.show()

print('Fitur paling berpengaruh terhadap Outcome:')
for rank, (feat, val) in enumerate(corr_with_outcome.items(), 1):
    print(f'  {rank}. {feat:<20}: {val:.3f}')

## 7. Scatter Plot Fitur Paling Penting

In [ ]:
# Top 4 fitur dari ranking korelasi
top_features = corr_with_outcome.head(4).index.tolist()
print(f'Top 4 fitur: {top_features}')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Scatter Plot Pasangan Fitur Paling Penting',
             fontsize=13, fontweight='bold')
axes = axes.flatten()

from itertools import combinations
pairs = list(combinations(top_features, 2))

for i, (fx, fy) in enumerate(pairs):
    ax = axes[i]
    for outcome, color, label in zip([0, 1], PALETTE, ['Tidak Diabetes', 'Diabetes']):
        subset = df_clean[df_clean['Outcome'] == outcome]
        ax.scatter(subset[fx], subset[fy], c=color, alpha=0.45,
                   s=25, label=label, edgecolors='none')
    ax.set_xlabel(fx, fontweight='bold')
    ax.set_ylabel(fy, fontweight='bold')
    ax.set_title(f'{fx} vs {fy}', fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Pair Plot (Gambaran Menyeluruh)

In [ ]:
# Pair plot untuk 4 fitur teratas + Outcome
# (menggunakan semua fitur akan terlalu lambat)
cols_pairplot = top_features + ['Outcome']
df_pair       = df_clean[cols_pairplot].copy()
df_pair['Status'] = df_pair['Outcome'].map({0: 'Tidak Diabetes', 1: 'Diabetes'})

g = sns.pairplot(
    df_pair.drop('Outcome', axis=1),
    hue='Status',
    palette={'Tidak Diabetes': '#378ADD', 'Diabetes': '#E24B4A'},
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 20},
    diag_kws={'alpha': 0.7}
)
g.fig.suptitle('Pair Plot — Top 4 Fitur Paling Berkorelasi dengan Diabetes',
               fontsize=13, fontweight='bold', y=1.01)

plt.savefig('plot_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Ringkasan Statistik per Kelompok

In [ ]:
# Statistik rata-rata per kelas
print('=== RATA-RATA FITUR PER KELAS (setelah imputasi) ===')
summary = df_clean.groupby('Outcome')[features].mean().round(2)
summary.index = ['Tidak Diabetes (0)', 'Diabetes (1)']
display(summary)

In [ ]:
# Uji statistik: apakah perbedaan antar kelas signifikan? (Mann-Whitney U test)
print('=== UJI STATISTIK (Mann-Whitney U) ===')
print(f'{"Fitur":<22} {"p-value":>12}  {"Signifikan?"}')
print('-' * 50)
for col in features:
    group0 = df_clean[df_clean['Outcome'] == 0][col]
    group1 = df_clean[df_clean['Outcome'] == 1][col]
    stat, p = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    sig = '✅ Ya (p < 0.05)' if p < 0.05 else '❌ Tidak'
    print(f'{col:<22} {p:>12.6f}  {sig}')

## 10. Simpan Dataset Bersih

In [ ]:
# Simpan dataset yang sudah dibersihkan
df_clean.to_csv('../data/diabetes_clean.csv', index=False)
print('✅ Dataset bersih disimpan ke: ../data/diabetes_clean.csv')
print(f'   Shape: {df_clean.shape}')
print()
print('File plot yang dihasilkan:')
plots = [
    'plot_before_imputation.png   — distribusi sebelum imputasi',
    'plot_distribusi_fitur.png    — distribusi per kelas',
    'plot_boxplot.png             — boxplot & outlier',
    'plot_heatmap_korelasi.png    — heatmap korelasi',
    'plot_korelasi_ranking.png    — ranking fitur',
    'plot_scatter.png             — scatter plot top fitur',
    'plot_pairplot.png            — pair plot'
]
for p in plots:
    print(f'  📊 {p}')

## 11. Kesimpulan EDA

### Temuan Utama

**Missing Values:**
- Ditemukan nilai `0` yang tidak valid di kolom: `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`
- Telah diganti dengan **median per kelas** (lebih akurat daripada median global)
- `Insulin` dan `SkinThickness` punya missing value terbanyak (>35%) — perlu perhatian lebih

**Fitur Paling Penting (dari korelasi):**
1. **Glucose** — korelasi tertinggi, sangat membedakan kedua kelas
2. **BMI** — pasien diabetes cenderung punya BMI lebih tinggi
3. **Age** — risiko meningkat seiring usia
4. **DiabetesPedigreeFunction** — riwayat keluarga berpengaruh

**Distribusi Dataset:**
- 500 pasien tidak diabetes (65.1%) vs 268 diabetes (34.9%)
- Dataset **sedikit imbalanced** → gunakan `class_weight='balanced'` di model atau perhatikan F1-Score

**Langkah Selanjutnya:**
- ✅ Dataset bersih siap untuk preprocessing (scaling)
- ✅ Pertimbangkan feature engineering dari fitur yang punya korelasi rendah
- ✅ Gunakan `diabetes_clean.csv` untuk training model ML
